# EXP-012: PLR-MLP + Inner Ensemble (n_ens=3, MLP only)

**근거:** RealMLP의 `n_ens=8` 기법 자체 구현. PLR이 큰 효과 입증(EXP-009/010)되어 PLR-MLP 위에 적용.

**Inner Ensemble:**
- 한 fold 안에서 n_ens개 모델 학습 후 val/test prediction 평균
- multi-seed (5-fold × seeds)와 다름 — inner는 fold-local 학습 stochasticity 분산 ↓
- n_ens=3 (RealMLP 8 → 3, 비용 통제)
- 각 ens seed: `SEED + ens_i * 1000`

**변경점 vs EXP-009 PLR-MLP**: fold 학습 loop 안에 ens loop 추가. 나머지 동일.

In [1]:
import pandas as pd
import numpy as np
import time, math
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

TARGET = 'PitNextLap'
CAT_COLS = ['Driver', 'Compound', 'Race']
NUM_COLS = ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
            'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
            'RaceProgress', 'Position_Change']
y = train[TARGET].astype(int).values

train_le = train.copy(); test_le = test.copy()
for c in CAT_COLS:
    le = LabelEncoder()
    train_le[c] = le.fit_transform(train_le[c].astype(str))
    seen = set(le.classes_)
    test_le[c] = test_le[c].astype(str).map(lambda v: le.transform([v])[0] if v in seen else -1)

drop_cols = ['id', TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]
X_le, X_test_le = train_le[feature_cols], test_le[feature_cols]

N_FOLDS, SEED = 5, 42
N_ENS = 3   # inner ensemble size
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(X_le, y))
print(f'Train: {train.shape}, Test: {test.shape}, N_ENS: {N_ENS}')

Train: (439140, 16), Test: (188165, 15), N_ENS: 3


## 1. PLR encoder + PLR-MLP (EXP-009과 동일)

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class PLREncoder(nn.Module):
    def __init__(self, n_numeric, k=16, hidden=8, sigma=2.33):
        super().__init__()
        self.freqs = nn.Parameter(torch.randn(n_numeric, k) * sigma)
        self.linear = nn.Linear(2 * k, hidden)
        self.act = nn.GELU()
    def forward(self, x_num):
        z = 2 * math.pi * x_num.unsqueeze(-1) * self.freqs.unsqueeze(0)
        p = torch.cat([torch.sin(z), torch.cos(z)], dim=-1)
        out = self.act(self.linear(p))
        return out.flatten(start_dim=1)


class PLRMLP(nn.Module):
    def __init__(self, cat_cards, emb_dims, n_numeric, plr_k=16, plr_hidden=8,
                 hidden=(256, 128, 64), dropout=0.2):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(card, dim) for card, dim in zip(cat_cards, emb_dims)])
        emb_total = sum(emb_dims)
        self.bn_num = nn.BatchNorm1d(n_numeric)
        self.plr = PLREncoder(n_numeric, k=plr_k, hidden=plr_hidden, sigma=2.33)
        plr_total = n_numeric * plr_hidden
        in_dim = emb_total + n_numeric + plr_total
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.body = nn.Sequential(*layers)
    def forward(self, x_cat, x_num):
        emb = torch.cat([emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.embs)], dim=1)
        raw_num = self.bn_num(x_num)
        plr_feat = self.plr(x_num)
        x = torch.cat([emb, raw_num, plr_feat], dim=1)
        return self.body(x).squeeze(1)

## 2. Cat / Num 인덱싱

In [3]:
cat_cards = []
X_cat_tr = np.zeros((len(X_le), len(CAT_COLS)), dtype=np.int64)
X_cat_te = np.zeros((len(X_test_le), len(CAT_COLS)), dtype=np.int64)
for i, c in enumerate(CAT_COLS):
    n_unique = X_le[c].max() + 1
    cat_cards.append(n_unique + 1)
    X_cat_tr[:, i] = X_le[c].values + 1
    te_vals = X_test_le[c].values + 1
    te_vals[te_vals < 0] = 0
    X_cat_te[:, i] = te_vals

X_num_tr_raw = X_le[NUM_COLS].values.astype(np.float32)
X_num_te_raw = X_test_le[NUM_COLS].values.astype(np.float32)

## 3. 5-fold × n_ens=3 학습

각 fold 안에서 3번 학습 (서로 다른 seed), val/test prediction 평균.

In [4]:
EMB_DIMS = [16, 4, 8]
BATCH_SIZE = 4096
MAX_EPOCHS = 80
PATIENCE = 20

oof_mlp = np.zeros(len(X_le))
test_mlp = np.zeros(len(X_test_le))
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(splits):
    scaler = StandardScaler()
    X_num_tr = scaler.fit_transform(X_num_tr_raw[tr_idx])
    X_num_va = scaler.transform(X_num_tr_raw[va_idx])
    X_num_te = scaler.transform(X_num_te_raw)

    Xc_tr = torch.tensor(X_cat_tr[tr_idx], dtype=torch.long)
    Xn_tr = torch.tensor(X_num_tr, dtype=torch.float32)
    yt_tr = torch.tensor(y[tr_idx], dtype=torch.float32)
    Xc_va = torch.tensor(X_cat_tr[va_idx], dtype=torch.long, device=device)
    Xn_va = torch.tensor(X_num_va, dtype=torch.float32, device=device)
    yt_va = y[va_idx]
    Xc_te = torch.tensor(X_cat_te, dtype=torch.long, device=device)
    Xn_te = torch.tensor(X_num_te, dtype=torch.float32, device=device)

    # Accumulate predictions across ens
    fold_va_pred = np.zeros(len(va_idx), dtype=np.float64)
    fold_te_pred = np.zeros(len(X_test_le), dtype=np.float64)

    for ens_i in range(N_ENS):
        ens_seed = SEED + ens_i * 1000
        torch.manual_seed(ens_seed)
        np.random.seed(ens_seed)

        train_loader = DataLoader(
            TensorDataset(Xc_tr, Xn_tr, yt_tr),
            batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
            num_workers=0, pin_memory=True,
            generator=torch.Generator().manual_seed(ens_seed),
        )

        model = PLRMLP(cat_cards, EMB_DIMS, len(NUM_COLS)).to(device)
        optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=MAX_EPOCHS)
        bce = nn.BCEWithLogitsLoss()

        best_auc = -1
        best_state = None
        wait = 0
        epochs_used = 0

        for epoch in range(MAX_EPOCHS):
            model.train()
            for xc, xn, yb in train_loader:
                xc = xc.to(device, non_blocking=True)
                xn = xn.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                optim.zero_grad()
                logit = model(xc, xn)
                loss = bce(logit, yb)
                loss.backward()
                optim.step()
            sched.step()

            model.eval()
            with torch.no_grad():
                va_logit = model(Xc_va, Xn_va).cpu().numpy()
            va_prob = 1 / (1 + np.exp(-va_logit))
            auc = roc_auc_score(yt_va, va_prob)
            epochs_used = epoch + 1

            if auc > best_auc + 1e-6:
                best_auc = auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    break

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            va_logit = model(Xc_va, Xn_va).cpu().numpy()
            te_logit = model(Xc_te, Xn_te).cpu().numpy()
        fold_va_pred += (1 / (1 + np.exp(-va_logit))) / N_ENS
        fold_te_pred += (1 / (1 + np.exp(-te_logit))) / N_ENS
        print(f'  fold {fold} ens {ens_i} (seed={ens_seed}): solo AUC={best_auc:.5f}, epochs={epochs_used}')

        del model, optim, sched
        torch.cuda.empty_cache()

    oof_mlp[va_idx] = fold_va_pred
    test_mlp += fold_te_pred / N_FOLDS
    fold_auc = roc_auc_score(yt_va, fold_va_pred)
    print(f'Fold {fold} ens-avg AUC: {fold_auc:.5f}')

    del Xc_va, Xn_va, Xc_te, Xn_te
    torch.cuda.empty_cache()

mlp_oof_auc = roc_auc_score(y, oof_mlp)
print(f'\nPLR-MLP + inner ens={N_ENS} OOF AUC: {mlp_oof_auc:.5f}')
print(f'  vs EXP-009 PLR-MLP 0.94486: {mlp_oof_auc-0.94486:+.5f}')
print(f'  elapsed: {time.time()-t0:.1f}s')

  fold 0 ens 0 (seed=42): solo AUC=0.94626, epochs=36
  fold 0 ens 1 (seed=1042): solo AUC=0.94565, epochs=34
  fold 0 ens 2 (seed=2042): solo AUC=0.94574, epochs=36
Fold 0 ens-avg AUC: 0.94861
  fold 1 ens 0 (seed=42): solo AUC=0.94366, epochs=38
  fold 1 ens 1 (seed=1042): solo AUC=0.94345, epochs=39
  fold 1 ens 2 (seed=2042): solo AUC=0.94415, epochs=34
Fold 1 ens-avg AUC: 0.94667
  fold 2 ens 0 (seed=42): solo AUC=0.94490, epochs=37
  fold 2 ens 1 (seed=1042): solo AUC=0.94495, epochs=38
  fold 2 ens 2 (seed=2042): solo AUC=0.94527, epochs=33
Fold 2 ens-avg AUC: 0.94783
  fold 3 ens 0 (seed=42): solo AUC=0.94431, epochs=40
  fold 3 ens 1 (seed=1042): solo AUC=0.94399, epochs=38
  fold 3 ens 2 (seed=2042): solo AUC=0.94409, epochs=34
Fold 3 ens-avg AUC: 0.94702
  fold 4 ens 0 (seed=42): solo AUC=0.94519, epochs=35
  fold 4 ens 1 (seed=1042): solo AUC=0.94482, epochs=37
  fold 4 ens 2 (seed=2042): solo AUC=0.94569, epochs=36
Fold 4 ens-avg AUC: 0.94811

PLR-MLP + inner ens=3 OOF AUC

## 4. Save predictions + 결론

In [5]:
np.save('../submissions/exp012_plr_inner_ens_oof.npy', oof_mlp)
np.save('../submissions/exp012_plr_inner_ens_test.npy', test_mlp)
sub = pd.DataFrame({'id': test['id'], 'PitNextLap': test_mlp})
sub.to_csv('../submissions/submission_exp012_plr_inner_ens_mlp.csv', index=False)
print(f'saved exp012 OOF/test arrays + submission CSV')
print(f'test pred mean: {test_mlp.mean():.4f}')
print()
print('=== EXP-012 결론 ===')
print(f'  EXP-009 PLR-MLP solo (n_ens=1) : 0.94486')
print(f'  EXP-012 PLR-MLP inner n_ens=3  : {mlp_oof_auc:.5f}')
print(f'  Δ (inner ensemble 효과)         : {mlp_oof_auc-0.94486:+.5f}')
print()
delta = mlp_oof_auc - 0.94486
if delta > 0.002:
    print(f'  → Inner ens 효과 큼. EXP-013 4-way 풀 학습 가치 큼')
elif delta > 0.0005:
    print(f'  → Inner ens 효과 중간. EXP-013 4-way 학습 가치 있음')
elif delta > -0.0005:
    print(f'  → Inner ens 효과 미미. 한계 효용 단계 — 마무리 검토')
else:
    print(f'  → 부정 효과. 본 데이터에 부적합')

saved exp012 OOF/test arrays + submission CSV
test pred mean: 0.2013

=== EXP-012 결론 ===
  EXP-009 PLR-MLP solo (n_ens=1) : 0.94486
  EXP-012 PLR-MLP inner n_ens=3  : 0.94763
  Δ (inner ensemble 효과)         : +0.00277

  → Inner ens 효과 큼. EXP-013 4-way 풀 학습 가치 큼
